# Toric-code-specific NES sampler

This notebook tests the flux-free initialization and runs the star/winding-loop Metropolis kernel. It is for the four toric-code ground states on a torus.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / 'Desktop' / 'Master Thesis' / 'nes_lattice_project'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
from nes_lattice.lattice import toric_code_move_masks, toric_code_plaquette_values
from nes_lattice.train import TrainConfig, train, save_history
from nes_lattice.plots import plot_history, plot_diagnostics, print_final


## 1. Static move check
Both star masks and winding-loop masks must preserve every plaquette value $B_p$.


In [ ]:
shape = (4, 4)
star_masks, loop_masks = toric_code_move_masks(shape)
base = np.ones((1, 2 * shape[0] * shape[1]), dtype=np.int8)

for label, masks in [('stars', star_masks), ('loops', loop_masks)]:
    moved = np.concatenate([base * (1 - 2 * mask)[None, :] for mask in masks], axis=0)
    bp = toric_code_plaquette_values(moved, shape)
    assert np.all(bp == 1), f'{label} did not preserve B_p=+1'

print('star masks:', star_masks.shape)
print('loop masks:', loop_masks.shape)
print('All star and winding-loop moves preserve B_p=+1.')


## 2. First 4x4 ground-state baseline
Use `k=1` first. The sampler uses 90% star moves and 10% winding loops.


In [ ]:
cfg = TrainConfig(
    shape=(4, 4),
    hamiltonian='toric_code',
    k=1,
    Je=1.0,
    Jm=1.0,

    model='vit',
    vit_patch_size=1,
    vit_d_model=64,
    vit_num_layers=4,
    vit_num_heads=4,
    vit_mlp_ratio=2,
    vit_use_positional_embeddings=False,

    steps=10_000,
    lr=5e-4,
    grad_clip=1.0,
    n_chains=256,
    n_samples=16,
    sweep_steps=32,
    burn_in=320,

    toric_loop_prob=0.10,
    toric_single_flip_prob=0.0,
    toric_cover_sectors=True,

    print_every=250,
    eval_exact_if_sites_leq=12,
    eval_chains=256,
    eval_samples=64,
    reference='auto',
    seed=0,
)

params, history = train(cfg)

save_path = PROJECT_ROOT / 'results' / 'sampled_nes_toric_4x4_k1_vit_toricmoves.json'
save_history(history, save_path, cfg)
print('saved to:', save_path)


## 3. Read the sampler diagnostics and plot


In [ ]:
final = history[-1]
for key in [
    'energies',
    'train_energy_estimator',
    'sampler_accept_rate',
    'sampler_star_accept_rate',
    'sampler_loop_accept_rate',
    'sampler_star_move_fraction',
    'sampler_loop_move_fraction',
    'invalid_bundle_fraction',
]:
    print(f'{key}:', final.get(key))

print_final(save_path)
fig, ax = plot_history(save_path)
fig


## 4. Fourfold ground space
Run this only after the `k=1` baseline is behaving reasonably.


In [ ]:
cfg4 = TrainConfig(
    shape=(4, 4),
    hamiltonian='toric_code',
    k=4,
    Je=1.0,
    Jm=1.0,

    model='vit',
    vit_patch_size=1,
    vit_d_model=64,
    vit_num_layers=4,
    vit_num_heads=4,
    vit_mlp_ratio=2,
    vit_use_positional_embeddings=False,

    steps=20_000,
    lr=2e-4,
    grad_clip=1.0,
    n_chains=256,
    n_samples=16,
    sweep_steps=32,
    burn_in=320,

    toric_loop_prob=0.10,
    toric_single_flip_prob=0.0,
    toric_cover_sectors=True,

    print_every=250,
    eval_exact_if_sites_leq=12,
    eval_chains=256,
    eval_samples=64,
    reference='auto',
    seed=1,
)

params4, history4 = train(cfg4)
save_path4 = PROJECT_ROOT / 'results' / 'sampled_nes_toric_4x4_k4_vit_toricmoves.json'
save_history(history4, save_path4, cfg4)
print('saved to:', save_path4)
